In [1]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.checkpoint.memory import MemorySaver

# 🔐 Load API keys
load_dotenv(".env")
google_api_key = os.getenv("GOOGLE_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

# 🔸 Initialize LLM (Gemini)
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",   # or "gemini-1.5-pro" for higher quality
    temperature=0,
    google_api_key=google_api_key,
)

# ✅ Tool 1: Simple QA Tool
qa_prompt = PromptTemplate.from_template("Answer clearly: {question}")

@tool
def simple_qa(question: str) -> str:
    """Answer the question without hallucination."""
    chain = qa_prompt | llm
    return chain.invoke({"question": question}).content

# ✅ Tool 2: Web Search Tool (Tavily)
tavily_search = TavilySearchResults(max_results=3)

@tool
def web_search(query: str) -> str:
    """Search the internet for current info."""
    return tavily_search.run(query)

tools = [simple_qa, web_search]

# 🧠 Memory: LangChain v1 uses a checkpointer instead of ConversationBufferMemory.
# It automatically stores the full message history per conversation "thread".
memory = MemorySaver()

agent_executor = create_agent(
    model=llm,
    tools=tools,
    checkpointer=memory,
)

# 🚀 Run a multi-turn conversation
# thread_id groups messages into the same conversation for memory to work
config = {"configurable": {"thread_id": "user-session-1"}}

def ask(query: str):
    print("\n🧑‍💻 User Query:", query)
    result = agent_executor.invoke(
        {"messages": [{"role": "user", "content": query}]},
        config=config,
    )
    response = result["messages"][-1].content
    print("\n🤖 Agent Response:", response)
    return response

if __name__ == "__main__":
    ask("What is LangGraph in LangChain?")
    ask("Can you summarize what you just told me in one sentence?")  # uses memory

C:\Users\Manikandan\AppData\Local\Temp\ipykernel_12724\2559062530.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults
C:\Users\Manikandan\AppData\Local\Temp\ipykernel_12724\2559062530.py:32: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_search = TavilySearchResults(max_results=3)



🧑‍💻 User Query: What is LangGraph in LangChain?

🤖 Agent Response: [{'type': 'text', 'text': "LangGraph, created by LangChain Inc., is a low-level orchestration framework and runtime specifically designed for building, managing, and deploying long-running, stateful AI agents. It utilizes graph-based architectures to model and manage complex generative AI agent workflows, allowing for dynamic processes that can include loops, branching, and revisiting previous states.\n\nWhile LangGraph is built by the creators of LangChain, it serves a different purpose and can be used independently. LangChain focuses on linear, step-by-step LLM workflows and rapid prototyping, providing higher-level abstractions and components. LangGraph, on the other hand, offers fine-grained control over agent orchestration, enabling the mixing of deterministic and LLM-driven steps, and providing explicit state management for more complex, adaptive, and multi-agent systems. LangChain's agent abstractions are built 

In [4]:
#1) Zero-Shot ReAct (simple + reliable default)
# Uses the `agent_executor` and `tools` already built with create_agent()
config = {"configurable": {"thread_id": "user-session-1"}}

result = agent_executor.invoke(
    {"messages": [{"role": "user", "content": "Summarize LangChain in 2 lines, then tell me who created it."}]},
    config=config,
)
print(result["messages"][-1].content)

LangChain is a framework that provides integrations and composable components to streamline the development of LLM applications, excelling in linear, step-by-step workflows and rapid prototyping. It offers prebuilt architectures for common LLM and tool-calling loops.

LangChain was created by LangChain Inc.


In [5]:
# Uses agent_executor (built with create_agent + checkpointer=memory) and tools from before
config = {"configurable": {"thread_id": "user-session-1"}}

result = agent_executor.invoke(
    {"messages": [{"role": "user", "content": "Summarize LangChain in 2 lines, then tell me who created it."}]},
    config=config,
)
print(result["messages"][-1].content)

[{'type': 'text', 'text': 'LangChain is a framework that provides integrations and composable components to streamline the development of LLM applications, excelling in linear, step-by-step workflows and rapid prototyping. It offers prebuilt architectures for common LLM and tool-calling loops.\n\nLangChain was created by LangChain Inc.', 'extras': {'signature': 'CuUBARFNMg9af95NPF+0DrYPBIw866KikQUpU1I2SzUOgduR3oNPBZCpy//b2vsJe+VI3TT2P/N3nqGByFROWUcAYCi4aPRNAcYfPMrTNBRDs8oKO1AZmw2Yu8voolk/Rs7c7APhz0ZvoX/ZM89zIgmdrkfz2uFeA5zJREZZnU+az2V8TvTR7d6cghNCV529zmXKcYgImklinhfijsCKaHcMRYzcEDq5K4dhr3d06NNI8EIrzJET3FlxrnRRlZP1PQepSN6c2hGXJwSwz0YjiYUDB4n7gC0/SE7frgL4whCBD2yeSfzIvw=='}}]


In [6]:
config = {"configurable": {"thread_id": "user-session-1"}}

result = agent_executor.invoke(
    {"messages": [{"role": "user", "content": "Plan a 3-step study path for LangChain."}]},
    config=config,
)
print(result["messages"][-1].content)

# result = agent_executor.invoke(
#     {"messages": [{"role": "user", "content": "Summarize LangChain in 2 lines, then tell me who created it."}]},
#     config=config,
# )
# print(result["messages"][-1].content)

[{'type': 'text', 'text': "Here's a 3-step study path for LangChain:\n\n1.  **Master the Fundamentals:** Begin by understanding LangChain's core concepts, including Language Models (LLMs), Prompts, Output Parsers, and basic Chains, to build simple, sequential LLM applications.\n2.  **Explore Agents and Tools:** Progress to learning about Agents, which enable LLMs to make decisions and use external Tools (like web search or custom functions) to perform tasks and interact with the environment.\n3.  **Dive into Advanced Features:** Study advanced topics such as Memory for conversational context, Retrieval for integrating external data, Callbacks for monitoring, and deploying LangChain applications with various integrations.", 'extras': {'signature': 'CswHARFNMg917YNW6CZO1jhw1q2gmCzPs9hRbBbRPaIeS5exa8Osm5VdY43jFnrkNBUZWP7SEtWrjUIMtJ0p/Gw4GytYPKsRapK78Xy4Jtf8hT/pfpKRY83EUS+v38H1U4OBg4dmCMUNKa3JnGHpNUSTga90yY4fMioA3tkLpw/lzrLXdvxwrJRQeTczxWpoTmIfFay5lupgqojWFcCBl2L6FHhKGdUyG+UmRH0FXc3f1YQ0Bp

In [7]:
from langchain_core.tools import StructuredTool, tool
from langchain.agents import create_agent
from pydantic import BaseModel

# ✅ Define the input schema using Pydantic
class TitleInput(BaseModel):
    topic: str
    tone: str = "concise"

# ✅ Define the tool function
def title_tool_fn(topic: str, tone: str = "concise") -> str:
    return f"{tone.title()} Title: {topic} in Practice"

# ✅ Create the structured tool (StructuredTool.from_function still works in v1)
title_tool = StructuredTool.from_function(
    name="TitleMaker",
    func=title_tool_fn,
    description="Generate a title given a topic and an optional tone.",
    args_schema=TitleInput
)

# ✅ Initialize the agent (v1 API — no AgentType needed)
agent_executor = create_agent(
    model=llm,           # assumes `llm` (e.g. ChatGoogleGenerativeAI) is already defined
    tools=[title_tool],
)

# ✅ Run the agent
result = agent_executor.invoke(
    {"messages": [{"role": "user", "content": "Make a friendly title about LangGraph tutorials."}]}
)
print("✅ Result:", result["messages"][-1].content)


# --------------------------------------------------------------------------
# Alternative: the @tool decorator can also handle multiple typed args with
# defaults directly from the function signature — no separate Pydantic class
# or StructuredTool.from_function needed, if you want something shorter:
# --------------------------------------------------------------------------
@tool
def title_maker(topic: str, tone: str = "concise") -> str:
    """Generate a title given a topic and an optional tone."""
    return f"{tone.title()} Title: {topic} in Practice"

# agent_executor = create_agent(model=llm, tools=[title_maker])

✅ Result: Here's a friendly title for LangGraph tutorials: "LangGraph Tutorials in Practice."


In [10]:
print("Tools bound to agent:", [t.name for t in tools])
test = agent_executor.invoke(
    {"messages": [{"role": "user", "content": "Use your Web_Search tool to look up 'LangGraph docs'."}]}
)
print(test["messages"][-1].content)

Tools bound to agent: ['simple_qa', 'web_search']
[{'type': 'text', 'text': 'I am sorry, I cannot use a "Web_Search" tool as I do not have access to it. I can use the `TitleMaker` tool to generate a title given a topic and an optional tone.', 'extras': {'signature': 'CtsBARFNMg9v97qjZEQDNOOcpx3Ue3Ofzv/DgGvjSKQRjyGC/KvjW3PlrMLa/gwwHIHchW8nHCJIYcsRnyeDJeZsTdty4d5VnVMCTa5XsxtA6jhGE5W95dsDHB5z/VmmTv1Mm1k6ZzIxAkmAqMkGr9y7DjOH6MqLktchdlFZo5DCN/L1SC20vX3Jmj36v0aey7TsTU8d86WRxvC73yTiyH8ADr2op/pat6UkQj/UNPMIwmRycMBoD+BIfvkKoMGfPlRT9YjBd+pavSCcBeCLI423Z/E+It+2TaS3Io5v'}}]


In [12]:
tools = [simple_qa, web_search]  # both tools this prompt needs
agent_executor = create_agent(model=llm, tools=tools)

print("Tools bound:", [t.name for t in tools])  # sanity check before calling

result = agent_executor.invoke(
    {"messages": [{"role": "user", "content": "Answer briefly from QA; also check the web for any updates."}]}
)
print(result["messages"][-1].content)

Tools bound: ['simple_qa', 'web_search']
[{'type': 'text', 'text': 'Please provide the question you would like me to answer.', 'extras': {'signature': 'CugBARFNMg+D0/zfvwyEmUQqD69jBP4jwe0C19OnIN6zFvwBMRHaQhVJbnI7/T0e0tMz40UH/nN/XVwgUuqAXvlcbX6o+/WwZrIPwbJRQ/MMoKKjoV7/OSWhnfKW7S4n1hXgZpNl9j8boXo0G5Zgs8mYvR6wUOYPRsiZwOV2RPV/r4P6CdsgqiTiHXs46CiheI9Q6vXalQAP4S9Q3Rky4ToXUW2l4eiDRZWdtywX3ofQcgz6aYzopR2NBt4FaHEOlFy66lRwnFmvhCd/LorAPxJ0yCHlScDQEhuwoCfV0s4UjvEt9eSWZTJQBQ=='}}]


In [13]:
from langchain_classic.agents import initialize_agent, AgentType
from langchain_classic.tools import Tool

search_tool = Tool(
    name="Intermediate Answer",  # still required by this specific algorithm
    func=TavilySearchResults(max_results=1).run,
    description="Use this tool to search for missing facts"
)

agent = initialize_agent(
    tools=[search_tool],
    llm=llm,
    agent=AgentType.SELF_ASK_WITH_SEARCH,
    verbose=True,
    handle_parsing_errors=True,
)
result = agent.run("When was the first LangChain release and who founded it?")

C:\Users\Manikandan\AppData\Local\Temp\ipykernel_12724\3456478469.py:10: LangChainDeprecationWarning: Use `langchain.agents.create_agent` for new applications. It provides a more flexible agent factory with middleware support, structured output, and integration with LangGraph for persistence, streaming, and human-in-the-loop workflows. Migration guide: https://docs.langchain.com/oss/python/migrate/langchain-v1
  agent = initialize_agent(
C:\Users\Manikandan\AppData\Local\Temp\ipykernel_12724\3456478469.py:17: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = agent.run("When was the first LangChain release and who founded it?")




> Entering new AgentExecutor chain...
No.
So the final answer is: The first LangChain release was in October 2022, and it was founded by Harrison Chase.

> Finished chain.
